# Drug-based CF

Proteins are users and drugs are items. Predict affinity using similar drugs with training scores for the same protein.


## Load saved splits

Use the existing training, validation and test pairs.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from utils.drug_cf import (DRUG, PROTEIN, fit_drug_cf, predict_drug_cf,
                             save_drug_cf, load_drug_cf)
from utils.evaluation import mean_squared_error, concordance_index, hit_at_k

PAIR_KEYS = [DRUG, PROTEIN]
MODELS = Path("saved_models")
splits = {
    name: {part: pd.read_csv(Path("data/splits") / f"{name}_{part}.csv",
                           float_precision="round_trip")
           for part in ["train", "val", "test"]}
    for name in ["davis", "kiba"]
}
for name, parts in splits.items():
    train = parts["train"]
    sets = {part: set(frame[PAIR_KEYS].itertuples(index=False, name=None))
            for part, frame in parts.items()}
    # check no pair appears in both training and validation
    assert sets["train"].isdisjoint(sets["val"])
    assert sets["train"].isdisjoint(sets["test"])
    assert sets["val"].isdisjoint(sets["test"])
    for part, frame in parts.items():
        assert len(sets[part]) == len(frame)
        assert np.isfinite(frame["affinity"]).all()
        for column in PAIR_KEYS:
            assert set(frame[column]) <= set(train[column])
    print(name.upper(), {part: len(frame) for part, frame in parts.items()})


DAVIS {'train': 18040, 'val': 3865, 'test': 3867}
KIBA {'train': 82359, 'val': 17648, 'test': 17650}


## Model

### Transposed affinity matrix

Rows are proteins and columns are drugs. Reuse item CF by swapping the ID columns. Missing training scores stay NaN; genuine zeros remain observed.


In [ ]:
# Adjusted similarity = Pearson similarity × overlap / (overlap + shrinkage)
models = {name: fit_drug_cf(parts["train"], shrinkage=10.0)
          for name, parts in splits.items()}
for name, model in models.items():
    values = model["values"]
    print(f"{name.upper()}: matrix {values.shape}, observed {np.isfinite(values).mean():.2%}, "
          f"observed zeros {np.count_nonzero(values == 0)}")
    assert np.count_nonzero(np.isfinite(values)) == len(splits[name]["train"])


DAVIS: matrix (379, 68), observed 70.00%, observed zeros 0
KIBA: matrix (229, 2068), observed 17.39%, observed zeros 1


### Drug similarity

Pearson correlation uses proteins observed for both drugs. Keep positive correlations and apply shrinkage: $s_{ij}=\max(0,r_{ij})n_{ij}/(n_{ij}+10)$.

Here minimum overlap counts shared proteins. Too few observations or constant scores give zero similarity.


In [6]:
for name, model in models.items():
    assert np.all(np.diag(model["similarities"]) == 0)
    assert np.isfinite(model["similarities"]).all()
    counts = model["common_counts"]
    upper = np.triu_indices(len(counts), 1)
    print(name.upper(), "median shared proteins:", float(np.median(counts[upper])))


DAVIS median shared proteins: 186.0
KIBA median shared proteins: 3.0


### Top-K neighbours

Exclude the query drug itself. Select up to K similar drugs with known training affinities for the query protein. Similarity ties use sorted SMILES.


### Affinity prediction

Predict the query drug’s training mean plus the similarity-weighted deviations from neighbour drug means. With no suitable neighbour, use the query drug’s mean; unknown drugs use the global mean.

Example: drug mean 6, neighbour drug mean 5, neighbour’s score for this protein 7. Prediction: $6+(7-5)=8$.


### Hyperparameter tuning

Try K = 5, 10, 20, 30 and minimum overlap = 2, 5, 10. Compute similarities once per dataset. Choose lowest validation MSE; ties favour smaller K, then smaller overlap. Test scores are not used.


In [5]:
tuning_rows = []
for name, model in models.items():
    validation = splits[name]["val"]
    for k in [5, 10, 20, 30]:
        for minimum in [2, 5, 10]:
            predictions, fallback = predict_drug_cf(
                model, validation[PAIR_KEYS], k=k, min_common=minimum)
            tuning_rows.append({"Dataset": name.upper(), "K": k,
                                "Min common": minimum,
                                "Val MSE": mean_squared_error(validation["affinity"], predictions),
                                "Val fallback %": 100 * fallback.mean()})
    trials = pd.DataFrame([row for row in tuning_rows if row["Dataset"] == name.upper()])
    best = trials.sort_values(["Val MSE", "K", "Min common"]).iloc[0]
    model.update(k=int(best["K"]), min_common=int(best["Min common"]),
                 dataset=name, score_scale="pKd" if name == "davis" else "KIBA score")
    print(name.upper(), "selected:", best.to_dict())
tuning = pd.DataFrame(tuning_rows)
print(tuning.to_string(index=False, float_format=lambda x: f"{x:.6f}"))


DAVIS selected: {'Dataset': 'DAVIS', 'K': 5, 'Min common': 2, 'Val MSE': 0.3229680045445596, 'Val fallback %': 0.0}
KIBA selected: {'Dataset': 'KIBA', 'K': 10, 'Min common': 2, 'Val MSE': 0.2090190029722499, 'Val fallback %': 2.5441976427923843}
Dataset  K  Min common  Val MSE  Val fallback %
  DAVIS  5           2 0.322968        0.000000
  DAVIS  5           5 0.322968        0.000000
  DAVIS  5          10 0.322968        0.000000
  DAVIS 10           2 0.327670        0.000000
  DAVIS 10           5 0.327670        0.000000
  DAVIS 10          10 0.327670        0.000000
  DAVIS 20           2 0.345012        0.000000
  DAVIS 20           5 0.345012        0.000000
  DAVIS 20          10 0.345012        0.000000
  DAVIS 30           2 0.353473        0.000000
  DAVIS 30           5 0.353473        0.000000
  DAVIS 30          10 0.353473        0.000000
   KIBA  5           2 0.210564        2.544198
   KIBA  5           5 0.212215        2.742520
   KIBA  5          10 0.228444   

## Evaluation

### Evaluation results

Reuse shared MSE, CI and Hit@K functions. Training metrics are in-sample: self-neighbours are excluded, but training scores contribute to means and similarities. Test predictions are evaluated only after selection.

Hit@K uses observed held-out candidates, includes singletons and counts any true-best target as a hit. Prediction ties use target IDs.


In [5]:
result_rows = []
saved_predictions = {}
for name, model in models.items():
    row = {"Dataset": name.upper(), "Model": "Drug CF", "Best K": model["k"],
           "Min common": model["min_common"], "Score scale": model["score_scale"]}
    saved_predictions[name] = {}
    for part_name, part in splits[name].items():
        predictions, fallback = predict_drug_cf(model, part[PAIR_KEYS])
        assert np.isfinite(predictions).all()
        saved_predictions[name][part_name] = (predictions, fallback)
        label = part_name.capitalize()
        row[f"{label} MSE"] = mean_squared_error(part["affinity"], predictions)
        row[f"{label} CI"] = concordance_index(part["affinity"], predictions)
        row[f"{label} fallback %"] = 100 * fallback.mean()
        if part_name != "train":
            for k in [1, 5]:
                hit = hit_at_k(part, predictions, k)
                row[f"{label} Hit@{k}"] = hit["hit_rate"]
                row[f"{label} eligible ligands"] = hit["eligible_ligands"]
    result_rows.append(row)
results = pd.DataFrame(result_rows)
print(results.to_string(index=False, float_format=lambda x: f"{x:.6f}"))


Dataset   Model  Best K  Min common Score scale  Train MSE  Train CI  Train fallback %  Val MSE   Val CI  Val fallback %  Val Hit@1  Val eligible ligands  Val Hit@5  Test MSE  Test CI  Test fallback %  Test Hit@1  Test eligible ligands  Test Hit@5
  DAVIS Drug CF       5           2         pKd   0.288737  0.865100          0.000000 0.322968 0.856107        0.000000   0.308824                    68   0.676471  0.312311 0.855041         0.000000    0.426471                     68    0.735294
   KIBA Drug CF      10           2  KIBA score   0.117061  0.890721          2.366469 0.209019 0.848886        2.544198   0.665989                  1967   0.934926  0.218736 0.847295         2.243626    0.654024                   1951    0.927217


### Saving and reloading models

Checkpoints contain training observations, IDs, similarities, overlap counts, means, hyperparameters and scale. Loaded models predict without notebook state.


In [6]:
for name, model in models.items():
    path = MODELS / f"{name}_drug_cf.npz"
    save_drug_cf(model, path)
    loaded = load_drug_cf(path)
    for part_name, part in splits[name].items():
        restored, flags = predict_drug_cf(loaded, part[PAIR_KEYS])
        original, original_flags = saved_predictions[name][part_name]
        np.testing.assert_allclose(restored, original, rtol=1e-12, atol=1e-12)
        np.testing.assert_array_equal(flags, original_flags)
    print(name.upper(), "checkpoint predictions match on all splits")


DAVIS checkpoint predictions match on all splits
KIBA checkpoint predictions match on all splits


### Results and limitations

DAVIS: K = 5, minimum overlap = 2, test MSE = 0.3123, CI = 0.8550, Hit@1 = 0.4265, Hit@5 = 0.7353.

KIBA: K = 10, minimum overlap = 2, test MSE = 0.2187, CI = 0.8473, Hit@1 = 0.6540, Hit@5 = 0.9272.

Test fallback: DAVIS 0%; KIBA 2.24%. Sparse overlaps make similarities less reliable. Training metrics are in-sample; singleton Hit@K always hits. Results cover known drugs and proteins.